# 02 — Business distribution

Describe business counts and category composition by district and planning area. Raw counts are not density measures and should not be interpreted as district attractiveness without suitable denominators.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from berlin_businesses.config import DATA_PROCESSED, REPORTS, FIGURES
from berlin_businesses.data import load_sources, coerce_coordinates
from berlin_businesses.analysis import count_by, category_shares_by_group
from berlin_businesses.visualize import plot_top_categories
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
df = coerce_coordinates(load_sources(ROOT / 'data' / 'raw'))
df.shape

In [ ]:
display(df.columns.tolist())
category_col = next((c for c in ['branch_top_level_desc', 'ihk_branch_desc', 'nace_desc'] if c in df.columns), None)
if category_col is None:
    raise KeyError('No recognized business category column found; inspect df.columns and update category_col.')
print('Category column:', category_col)

In [ ]:
if 'Bezirk' in df.columns:
    district_counts = count_by(df, 'Bezirk', dropna=False)
    display(district_counts)
    district_counts.to_csv(REPORTS / 'district_counts.csv', index=False)
else:
    print('Bezirk column not found in source data.')

In [ ]:
category_counts = count_by(df, category_col, dropna=False)
display(category_counts.head(20))
category_counts.to_csv(REPORTS / 'business_category_counts.csv', index=False)
fig, ax = plot_top_categories(category_counts, category_col, top_n=15, output=FIGURES / 'top_business_categories.png')
plt.show()

In [ ]:
if 'Bezirk' in df.columns:
    shares = category_shares_by_group(df, 'Bezirk', category_col)
    display(shares.sort_values(['Bezirk', 'share_within_group'], ascending=[True, False]).groupby('Bezirk').head(10))
    shares.to_csv(REPORTS / 'category_shares_by_district.csv', index=False)

## Interpretation guardrails

- District totals reflect the number of records in the source, not population-adjusted or area-adjusted density.
- The selected category field must be documented: `branch_top_level_desc`, `ihk_branch_desc`, and `nace_desc` represent different classification levels.
- When combining monthly exports, account for repeat listings across months.